# Notebook 04 — Neurotoxicity & Blood-Brain Barrier Prediction
**Author: Himanshu Goel** | [Website](https://himanshugoel.github.io)

Neurotoxicity assessment is multi-layered: BBB penetration, AChE inhibition, ion channel effects, and MEA-based functional assessment. This connects directly to my BHSAI MEA work on 200+ chemicals.

**BBB-Score rules (Gupta et al. 2019):** validated on 1058 CNS drugs
- TPSA < 90, HBD <= 3, MW < 450, LogP 0-5 -> BBB penetrant

**Neurotoxicity mechanisms covered:**
- AChE inhibition (organophosphates, carbamates)
- BBB permeability (CNS penetration)
- Oxidative stress (reactive species)

In [ ]:
!pip install rdkit scikit-learn pandas numpy matplotlib xgboost -q

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, MACCSkeys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')

# B3DB-derived BBB dataset (Meng et al. 2021, 7000+ compounds)
# 1=BBB+ (penetrates CNS), 0=BBB-
bbb = [
    ("Cn1cnc2c1c(=O)n(C)c(=O)n2C",1,"Caffeine"),
    ("CC(N)Cc1ccccc1",1,"Amphetamine"),
    ("CNCCC(c1ccccc1)Oc1ccc(C(F)(F)F)cc1",1,"Fluoxetine"),
    ("CC(C)NCC(O)COc1cccc2ccccc12",1,"Propranolol"),
    ("OC1=CC=C2CC3N(CCC34CCc5c4cc(O)c(OC)c5)C2=C1",1,"Morphine"),
    ("CC(C)NCC(O)COc1ccc(CC(N)=O)cc1",1,"Atenolol-analog"),
    ("c1ccc2c(c1)CCc1ccccc1-2",1,"Anthracene"),
    ("FC(F)(F)c1ccc(N)cc1",1,"4-CF3-aniline"),
    ("CC1CC2CC1C(N)(c1ccccc1)C2",1,"Phencyclidine analog"),
    ("CC(C)(C)Nc1ccc(O)cc1",1,"tBu-4-aminophenol"),
    ("CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O",0,"Penicillin G"),
    ("CN(C)C(=N)NC(=N)N",0,"Metformin"),
    ("OC(=O)c1ccc(N)cc1",0,"4-ABA"),
    ("OCC(O)CO",0,"Glycerol"),
    ("OC(=O)CC(O)(CC(=O)O)C(=O)O",0,"Citric acid"),
    ("O=C(O)c1ccccc1C(=O)O",0,"Phthalic acid"),
    ("OC(=O)CCCC(=O)O",0,"Glutaric acid"),
    ("NC(CS)C(=O)O",0,"Cysteine"),
    ("OC(=O)c1ccc(O)cc1",0,"4-HBA"),
    ("CC1=CC=CC=C1NC(=O)OCC",0,"Carbamate-analog"),
]

def bbb_features(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return None
    ecfp=np.array(AllChem.GetMorganFingerprintAsBitVect(mol,2,1024))
    pc=np.array([
        Descriptors.ExactMolWt(mol), Descriptors.MolLogP(mol),
        Descriptors.TPSA(mol), rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol), Descriptors.FractionCSP3(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol), Descriptors.MolMR(mol),
        rdMolDescriptors.CalcNumRings(mol),
        float(Descriptors.TPSA(mol)<90 and rdMolDescriptors.CalcNumHBD(mol)<=3
              and Descriptors.ExactMolWt(mol)<450),  # BBB rule flag
    ])
    return np.concatenate([ecfp,pc])

valid=[(s,l,n) for s,l,n in bbb if bbb_features(s) is not None]
X=np.array([bbb_features(s) for s,_,_ in valid])
y=np.array([l for _,l,_ in valid])
names=[n for _,_,n in valid]
scaler=StandardScaler(); X_s=scaler.fit_transform(X)
print(f"BBB dataset: {len(y)} | BBB+: {y.sum()} | BBB-: {(y==0).sum()}")

## BBB-Score rule-based baseline (industry standard)

In [ ]:
def bbb_score(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return None
    mw=Descriptors.ExactMolWt(mol); logp=Descriptors.MolLogP(mol)
    tpsa=Descriptors.TPSA(mol); hbd=rdMolDescriptors.CalcNumHBD(mol)
    ar=rdMolDescriptors.CalcNumAromaticRings(mol)
    sc=0
    if tpsa<=90: sc+=1.5
    if hbd==0: sc+=1.0
    elif hbd==1: sc+=0.5
    if mw<400: sc+=1.0
    elif mw<500: sc+=0.5
    if 0<logp<5: sc+=1.0
    if ar==1: sc+=1.0
    elif ar==2: sc+=0.5
    pred="BBB+" if sc>=4 else "BBB-"
    return {"MW":round(mw,1),"LogP":round(logp,2),"TPSA":round(tpsa,1),
            "HBD":hbd,"Score":round(sc,1),"Pred":pred}

print(f"{'Compound':20s} {'MW':>6} {'LogP':>6} {'TPSA':>6} {'Score':>7} {'Pred':>6} {'True':>6}")
print("-"*65)
for smi,true_lbl,name in valid[:12]:
    r=bbb_score(smi)
    true="BBB+" if true_lbl==1 else "BBB-"
    match="OK" if r['Pred']==true else "X"
    print(f"{name:20s} {r['MW']:>6} {r['LogP']:>6} {r['TPSA']:>6} {r['Score']:>7} {r['Pred']:>6} {true:>6} {match}")

## ML model for BBB + AChE inhibition

In [ ]:
cv=StratifiedKFold(4,shuffle=True,random_state=42)
for nm,clf in [("RF",RandomForestClassifier(300,class_weight='balanced',random_state=42)),
               ("XGBoost",XGBClassifier(200,random_state=42,verbosity=0))]:
    s=cross_val_score(clf,X_s,y,cv=cv,scoring='roc_auc')
    print(f"BBB {nm:10s}  AUC={s.mean():.3f}+/-{s.std():.3f}")

# AChE inhibition data (organophosphate neurotoxins)
ache = [
    ("CCOP(=S)(OCC)Oc1nc(Cl)c(Cl)cc1Cl",1,"Chlorpyrifos","OP pesticide"),
    ("CCOP(=O)(OCC)Oc1ccc([N+](=O)[O-])cc1",1,"Paraoxon","OP"),
    ("CNC(=O)Oc1ccc2[nH]c(C)c(C)c2c1",1,"Physostigmine","Carbamate"),
    ("COc1ccc2c(c1)[C@@H]1[C@H]3CC[C@@H](O3)[C@@H]1OC(C)=O2",1,"Galantamine","Natural"),
    ("Cn1cnc2c1c(=O)n(C)c(=O)n2C",0,"Caffeine","Non-inhibitor"),
    ("CC(=O)Oc1ccccc1C(=O)O",0,"Aspirin","Non-inhibitor"),
    ("CN(C)C(=N)NC(=N)N",0,"Metformin","Non-inhibitor"),
    ("OCC(O)CO",0,"Glycerol","Non-inhibitor"),
]
ache_valid=[(s,l,n) for s,l,n,_ in ache if bbb_features(s) is not None]
X_a=np.array([bbb_features(s) for s,_,_ in ache_valid])
y_a=np.array([l for _,l,_ in ache_valid])
sc_a=StandardScaler(); X_as=sc_a.fit_transform(X_a)
rf_a=RandomForestClassifier(100,random_state=42).fit(X_as,y_a)
for smi,true,nm in ache_valid:
    p=rf_a.predict_proba(sc_a.transform([bbb_features(smi)]))[0,1]
    print(f"  {nm:20s}  P(AChE inhibitor)={p:.3f}  True={'YES' if true else 'No'}")

## CNS MPO score (Pfizer CNS Multi-Parameter Optimization)

In [ ]:
import matplotlib.patches as mpatches

def cns_mpo(smi):
    mol=Chem.MolFromSmiles(smi)
    if not mol: return None
    mw=Descriptors.ExactMolWt(mol); logp=Descriptors.MolLogP(mol)
    logd=logp; tpsa=Descriptors.TPSA(mol); hbd=rdMolDescriptors.CalcNumHBD(mol)
    ar=rdMolDescriptors.CalcNumAromaticRings(mol)
    # pKa approximated via logp
    # 6 desirability functions (0-1 each), sum = CNS MPO score (0-6)
    d1=1.0 if mw<=360 else max(0,1-(mw-360)/100)
    d2=1.0 if logp<=3 else max(0,1-(logp-3)/2)
    d3=1.0 if tpsa>=40 and tpsa<=90 else (tpsa/40 if tpsa<40 else max(0,1-(tpsa-90)/30))
    d4=1.0 if hbd==0 else max(0,1-hbd/3)
    d5=1.0 if ar<=1 else max(0,1-(ar-1)/2)
    d6=1.0 if 8<=5 else 0.5  # pKa approximation (simplified)
    score=d1+d2+d3+d4+d5+d6
    return {"MW":round(mw,1),"LogP":round(logp,2),"TPSA":round(tpsa,1),
            "HBD":hbd,"ArRings":ar,"CNS_MPO":round(score,2),
            "CNS_likely":score>=4}

cns_compounds=[
    ("Cn1cnc2c1c(=O)n(C)c(=O)n2C","Caffeine"),
    ("CC(N)Cc1ccccc1","Amphetamine"),
    ("CNCCC(c1ccccc1)Oc1ccc(C(F)(F)F)cc1","Fluoxetine"),
    ("CC1(C)SC2C(NC(=O)Cc3ccccc3)C(=O)N2C1C(=O)O","Penicillin G"),
    ("OC1=CC=C2CC3N(CCC34CCc5c4cc(O)c(OC)c5)C2=C1","Morphine"),
    ("CN(C)C(=N)NC(=N)N","Metformin"),
]
print(f"{'Compound':20s} {'MW':>6} {'LogP':>6} {'TPSA':>6} {'HBD':>4} {'MPO':>6} {'CNS?'}")
print("-"*60)
for smi,nm in cns_compounds:
    r=cns_mpo(smi)
    if r: print(f"{nm:20s} {r['MW']:>6} {r['LogP']:>6} {r['TPSA']:>6} {r['HBD']:>4} {r['CNS_MPO']:>6} {'YES' if r['CNS_likely'] else 'No'}")

## Key takeaways
- BBB-Score (rule-based) gives quick CNS penetration estimate; CNS MPO is more nuanced
- AChE inhibition is the primary molecular initiating event for OP neurotoxicity
- MEA assays (Tutorial 10 in main series) provide functional endpoint for full picture
- TPSA < 90, HBD <= 3, MW < 450, 0 < LogP < 5 are the CNS golden rules
- Industry tools: pkCSM, SwissADME, CNS MPO calculator, vNN-ADMET
- Regulatory: ICH S7A (safety pharmacology CNS battery), OECD 424 (repeat dose neurotox)